# Day 3-02｜同じ選手の ID をフレーム間で維持する：YOLO と ByteTrack

[Colab で開く](https://colab.research.google.com/github/henry753951/basketball-hackathon-course/blob/main/day3/d3_02_yolo_bytetrack_tracking_ja.ipynb)

[English](d3_02_yolo_bytetrack_tracking_en.ipynb) | [Japanese](d3_02_yolo_bytetrack_tracking_ja.ipynb) | [Traditional Chinese](d3_02_yolo_bytetrack_tracking.ipynb)


> 前の単元では 2 枚の画像を比較した。ここでは検出を、より長い時系列としてつなぐ。  
> 画面の `#7` は背番号でも実名でもない。この動画内で一時的に割り当てられた `track_id` である。

## 学習目標

- 矩形を見つける検出器と、時間的な関係をつなぐ追跡器の役割を区別する。
- 矩形、`track_id`、直近の軌跡を表示した動画を出力する。
- フレームごとの追跡結果を DataFrame にまとめる。
- 観測フレーム数と信頼度から ID の安定性を調べる。

## 用語

- **複数物体追跡（Multi-object tracking）**：複数の対象の ID を時間にわたって維持すること。
- **Track ID**：動画内で追跡器が一時的に付ける識別子。再実行すると異なる場合がある。
- **軌跡（Trajectory / trace）**：同じ追跡対象の直近の位置を結んだ線。
- **ID の入れ替わり（ID switch）**：交差後に別の選手へ ID をつなぐなど、識別を誤ること。


## この単元の位置付け

- 既存の ByteTrack を使い、アルゴリズム全体の再実装は行わない。
- 入力する検出、出力される追跡、失敗する条件の関係を確認する。
- 処理が遅ければ `MAX_FRAMES` を `30` にし、流れを確認してから増やす。


## 演習の流れ

1. 動画、モデル、推論パラメータを設定する。
2. `YOLO detection -> ByteTrack association -> 可視化` を実行する。
3. 同じ ID の矩形と短い軌跡が連続しているかを見る。
4. JSON／DataFrame を読み、各行が 1 フレームの 1 追跡対象を表すことを理解する。
5. 各 ID の継続期間を集計し、短すぎる追跡や入れ替わりの疑いを探す。


In [1]:
import os
os.environ["BASKETBALL_COURSE_LANGUAGE"] = "ja"

# リポジトリを探し、環境を準備して共通モジュールを読み込みます。
from pathlib import Path
import subprocess
import sys

DRIVE_MOUNTED = False
if "google.colab" in sys.modules:
    from google.colab import drive

    try:
        drive.mount("/content/drive")
        DRIVE_MOUNTED = True
    except NotImplementedError:
        print("この Colab ランタイムでは Drive をマウントできないため、ローカルの /content を使います。")

COURSE_ROOT_HINT = (
    Path("/content/drive/MyDrive/basketball_hackathon/course")
    if DRIVE_MOUNTED
    else next(
        (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
         if (p / "src" / "course_setup.py").exists()),
        Path("/content/basketball_hackathon/course"),
    )
)
if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists() and "google.colab" in sys.modules:
    COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/henry753951/basketball-hackathon-course.git", str(COURSE_ROOT_HINT)
    ], check=True)
if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_repo  # noqa: E402

COURSE_ROOT = bootstrap_course_repo(COURSE_ROOT_HINT, mount_drive=False)


教材ルート： /Users/henry753951/Developer/sandbox/basketball-hackathon-course
素材フォルダ： /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets
共通モジュール： /Users/henry753951/Developer/sandbox/basketball-hackathon-course/src


## Step 1｜入力、出力、処理範囲を設定する

`START_FRAME` は開始位置、`MAX_FRAMES` は処理するフレーム数を決める。短い動画から始めると、パラメータの比較を早く繰り返せる。


In [2]:
# パスと調整するパラメータをまとめます。
import pandas as pd

from src.video_utils import display_video_in_notebook
from src.yolo_utils import (
    detector_model_path,
    first_reference_video,
    write_bytetrack_preview_video,
)

VIDEO_PATH = first_reference_video(COURSE_ROOT)
MODEL_PATH = detector_model_path(COURSE_ROOT)
OUTPUT_PATH = COURSE_ROOT / "assets" / "results" / "d3_02_bytetrack_preview.mp4"
START_FRAME = 0
MAX_FRAMES = 120
CONF = 0.25
IMGSZ = 960

print("video:", VIDEO_PATH)
print("model:", MODEL_PATH)
print("output:", OUTPUT_PATH)
print(f"frame range: {START_FRAME} .. {START_FRAME + MAX_FRAMES - 1}")


video: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/raw/reference_videos/boston-celtics-new-york-knicks-game-1-q1-01.54-01.48.mp4
model: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/models/detectors/yolo26n_basketball_player_best.pt
output: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d3_02_bytetrack_preview.mp4
frame range: 0 .. 119


## Step 2｜ByteTrack を実行し、直近の軌跡を描く

共通関数は各フレームで次の順に処理する。

```text
画像 -> YOLO の矩形 -> ByteTrack の ID 更新 -> 矩形・ラベル・直近軌跡の描画 -> 動画／JSON の保存
```

軌跡は移動方向を見るための短い履歴で、試合全体の経路ではない。`#12 player 0.87` は、順に track ID、クラス、検出の信頼度を表す。


In [3]:
# 追跡の確認動画とフレームごとの記録を作ります。
preview_video, records = write_bytetrack_preview_video(
    video_path=VIDEO_PATH,
    model_path=MODEL_PATH,
    output_path=OUTPUT_PATH,
    max_frames=MAX_FRAMES,
    conf=CONF,
    imgsz=IMGSZ,
    start_frame=START_FRAME,
)
preview_json = preview_video.with_suffix(".json")

print("preview video:", preview_video)
print("preview json:", preview_json)
print("number of track records:", len(records))
display_video_in_notebook(preview_video, loop=True)


$ /opt/homebrew/bin/ffmpeg -y -i /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d3_02_bytetrack_preview.mp4 -an -vcodec libx264 -pix_fmt yuv420p -movflags +faststart -preset veryfast -crf 23 /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d3_02_bytetrack_preview.notebook.mp4


preview video: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d3_02_bytetrack_preview.mp4
preview json: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d3_02_bytetrack_preview.json
number of track records: 1893


### 動画の確認項目

- 同じ選手が動いても、矩形の `#ID` はおおむね変わらないか。
- 遮蔽後に元の ID が戻るか、新しい ID になるか、別の選手と入れ替わるか。
- 軌跡の急な飛びは、矩形の揺れでも誤対応でも起こる。


## Step 3｜動画のラベルを表として読む

DataFrame の各行は、あるフレームのある `track_id` の 1 回の観測であり、選手全体の要約ではない。同じ ID が複数行に現れることで軌跡データになる。


In [4]:
# 列の順序を固定し、追跡記録を項目ごとに読みます。
track_columns = [
    "frame",       # 元動画のフレーム番号。
    "track_id",    # 追跡器が一時的に割り当てるフレーム間の ID。
    "class_id",    # モデル内部のクラス番号。
    "class_name",  # player などのクラス名。
    "confidence",  # この矩形に対する検出器の信頼度。
    "bbox_xyxy",   # [left, top, right, bottom]
]
tracks = pd.DataFrame(records, columns=track_columns)
tracks.head(10)


,frame,track_id,class_id,class_name,confidence,bbox_xyxy
0,0,1,3,player,0.931542,"[545.8660888671875, 469.2069091796875, 627.810..."
1,0,2,3,player,0.916845,"[1314.0743408203125, 610.4462280273438, 1387.2..."
2,0,3,3,player,0.915130,"[1399.1689453125, 410.537353515625, 1481.96936..."
3,0,4,3,player,0.908453,"[1251.930908203125, 401.5520935058594, 1390.92..."
4,0,5,3,player,0.894809,"[374.1379699707031, 500.90692138671875, 501.01..."
5,0,6,8,referee,0.893409,"[997.5321655273438, 699.8613891601562, 1096.02..."
6,0,7,3,player,0.872467,"[1250.6168212890625, 535.5084838867188, 1325.2..."
7,0,8,3,player,0.871733,"[568.2689819335938, 219.78465270996094, 697.77..."
8,0,9,8,referee,0.869683,"[66.15065002441406, 175.08575439453125, 130.10..."
9,0,10,3,player,0.848960,"[521.8480224609375, 159.7144317626953, 625.388..."


## Step 4｜追跡単位で品質を確認する

- `frames_seen` が小さい：誤検出の可能性も、選手の入退場の可能性もある。
- `first_frame` から `last_frame` が長く、`frames_seen` が少ない：途中で見落としが繰り返されている可能性がある。
- `mean_confidence` が低い：入力矩形の信頼性が低く、後の位置分析に注意が必要である。

これらは疑わしい追跡を探す手掛かりであり、単独で追跡の正しさを証明するものではない。


In [5]:
# フレームごとの記録を track_id ごとに 1 行に集計します。
summary_columns = [
    "track_id", "frames_seen", "first_frame", "last_frame",
    "time_span_frames", "coverage_ratio", "mean_confidence",
]
if tracks.empty:
    track_summary = pd.DataFrame(columns=summary_columns)
else:
    track_summary = (
        tracks.groupby("track_id", dropna=False)
        .agg(
            frames_seen=("frame", "count"),
            first_frame=("frame", "min"),
            last_frame=("frame", "max"),
            mean_confidence=("confidence", "mean"),
        )
        .reset_index()
    )
    track_summary["time_span_frames"] = (
        track_summary["last_frame"] - track_summary["first_frame"] + 1
    )
    track_summary["coverage_ratio"] = (
        track_summary["frames_seen"] / track_summary["time_span_frames"]
    )
    track_summary = track_summary.sort_values(
        ["frames_seen", "first_frame"], ascending=[False, True]
    )

track_summary[summary_columns].head(15)


,track_id,frames_seen,first_frame,last_frame,time_span_frames,coverage_ratio,mean_confidence
1,2,120,0,119,120,1.000000,0.834083
5,6,120,0,119,120,1.000000,0.878798
3,4,115,0,119,120,0.958333,0.836258
0,1,114,0,119,120,0.950000,0.860714
2,3,113,0,119,120,0.941667,0.864025
8,9,111,0,112,113,0.982301,0.878783
21,22,67,10,76,67,1.000000,0.821516
14,15,65,0,69,70,0.928571,0.773104
4,5,60,0,63,64,0.937500,0.872554
9,10,55,0,73,74,0.743243,0.766611


## 出力と次の単元

- `assets/results/d3_02_bytetrack_preview.mp4`：ID、矩形、短い軌跡を表示した動画。
- `assets/results/d3_02_bytetrack_preview.json`：フレームごとの追跡記録。
- `track_summary`：各 ID の継続期間、観測範囲、平均信頼度。

以降の単元では、ユニフォームの色でチーム分けし、矩形の足元、追跡、BEV 投影を統合する。
